# RAVDESS Dataset Explorer
Loads `xbgoose/ravdess` from HuggingFace and prints statistics.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from collections import Counter

print('Imports OK')

In [ ]:
ds = load_dataset('xbgoose/ravdess')
print(ds)

## Overview

In [ ]:
split_name = list(ds.keys())[0]
split = ds[split_name]

print(f'Split      : {split_name}')
print(f'Num samples: {len(split):,}')
print(f'Columns    : {split.column_names}')
print(f'Features   :')
for col, feat in split.features.items():
    print(f'  {col}: {feat}')

## Sample Entry

In [ ]:
ex = split[0]
print('Keys:', list(ex.keys()))
print()
for k, v in ex.items():
    if k == 'audio':
        print(f'audio:')
        print(f'  sampling_rate : {v["sampling_rate"]} Hz')
        print(f'  array shape   : {np.array(v["array"]).shape}')
        print(f'  duration      : {len(v["array"]) / v["sampling_rate"]:.2f}s')
    else:
        print(f'{k}: {v}')

## Label Distribution

In [ ]:
# Detect label column
label_col = next((c for c in ['label', 'emotion', 'labels', 'target'] if c in split[0]), None)
print(f'Label column: "{label_col}"')

all_labels = split[label_col]

# Resolve label names
RAVDESS_NAMES = ['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']
feat = split.features.get(label_col)
if hasattr(feat, 'names'):
    label_names = feat.names
else:
    min_l = min(all_labels)
    label_names = RAVDESS_NAMES[:max(all_labels) - min_l + 1]

counts = Counter(all_labels)
print(f'\n{"Label":>5}  {"Name":<12}  {"Count":>6}  {"Share":>7}')
print('-' * 36)
for idx in sorted(counts):
    name  = label_names[idx] if idx < len(label_names) else str(idx)
    share = counts[idx] / len(all_labels) * 100
    print(f'{idx:>5}  {name:<12}  {counts[idx]:>6}  {share:>6.1f}%')

In [ ]:
names  = [label_names[i] if i < len(label_names) else str(i) for i in sorted(counts)]
values = [counts[i] for i in sorted(counts)]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(names, values, color=sns.color_palette('Set2', len(names)))
ax.bar_label(bars, padding=3)
ax.set_title('Sample Count per Emotion Class')
ax.set_xlabel('Emotion')
ax.set_ylabel('Count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('label_distribution.png', dpi=150)
plt.show()

## Audio Duration Statistics

In [ ]:
print('Computing durations (this may take a moment)...')
durations = [
    len(ex['audio']['array']) / ex['audio']['sampling_rate']
    for ex in split
]
durations = np.array(durations)

print(f'Min duration    : {durations.min():.2f}s')
print(f'Max duration    : {durations.max():.2f}s')
print(f'Mean duration   : {durations.mean():.2f}s')
print(f'Median duration : {np.median(durations):.2f}s')
print(f'Std deviation   : {durations.std():.2f}s')
print(f'Sampling rate   : {split[0]["audio"]["sampling_rate"]} Hz')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(durations, bins=30, color='steelblue', edgecolor='white', linewidth=0.5)
ax.axvline(durations.mean(),   color='coral',    linestyle='--', label=f'Mean {durations.mean():.2f}s')
ax.axvline(np.median(durations), color='seagreen', linestyle='--', label=f'Median {np.median(durations):.2f}s')
ax.set_title('Distribution of Audio Clip Durations')
ax.set_xlabel('Duration (seconds)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig('duration_distribution.png', dpi=150)
plt.show()

## Duration by Emotion Class

In [ ]:
dur_by_class = {name: [] for name in label_names}
for ex, dur in zip(split, durations):
    name = label_names[ex[label_col]] if ex[label_col] < len(label_names) else str(ex[label_col])
    dur_by_class[name].append(dur)

print(f'{"Emotion":<12}  {"Count":>6}  {"Min":>6}  {"Max":>6}  {"Mean":>6}')
print('-' * 44)
for name, durs in dur_by_class.items():
    if durs:
        d = np.array(durs)
        print(f'{name:<12}  {len(d):>6}  {d.min():>5.2f}s  {d.max():>5.2f}s  {d.mean():>5.2f}s')

In [ ]:
data   = [dur_by_class[n] for n in label_names if dur_by_class.get(n)]
labels = [n for n in label_names if dur_by_class.get(n)]

fig, ax = plt.subplots(figsize=(10, 5))
bp = ax.boxplot(data, patch_artist=True, tick_labels=labels)
colors = sns.color_palette('Set2', len(labels))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
ax.set_title('Audio Duration Distribution per Emotion Class')
ax.set_xlabel('Emotion')
ax.set_ylabel('Duration (seconds)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('duration_by_class.png', dpi=150)
plt.show()